# 6.2 Vector Database

A vector database is a specialized system for storing and searching high-dimensional data, such as embeddings from text, images, or audio. In Retrieval-Augmented Generation (RAG), vector databases enable efficient similarity search to retrieve relevant information based on semantic meaning, improving the quality and relevance of generated responses.

In [ ]:
%pip install fastapi uvicorn sentence-transformers faiss-cpu
%pip install tf-keras

In [ ]:
import joblib
import glob
import faiss
import pandas as pd
import numpy as np

from fastapi import FastAPI
from uvicorn import Config, Server

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

from sentence_transformers import SentenceTransformer

## **TASK: Create a Simple Vector Database using TF-IDF**

In this task, you will build a basic vector database for text data using TF-IDF embeddings and nearest neighbor search.

Steps:
1. Prepare a small collection of text documents.
2. Convert the documents into TF-IDF vectors using scikit-learn.
3. Build a nearest neighbor index to enable similarity search over the vectors.
4. Implement a search function to retrieve the most similar documents for a given query.
5. Save the vectorizer, index, and data for later use.

This workflow demonstrates how vector databases can store and search text embeddings, enabling semantic retrieval for applications like RAG.

In [ ]:
# 1. Prepare a small collection of text documents.
data = [
    "Machine learning is a field of artificial intelligence",
    "Neural networks are inspired by the brain",
    "FastAPI is a modern web framework for building APIs",
    "Vector databases enable efficient similarity search",
    "Text embeddings convert documents into numerical vectors",
    "Supervised learning uses labeled data",
    "Unsupervised learning finds patterns without labels",
    "Reinforcement learning trains agents through rewards",
    "Python is a popular language for data science",
    "APIs allow applications to communicate with each other"
]

df = pd.DataFrame({'text': data})
df

#### TF-IDF (Term Frequency – Inverse Document Frequency)

TF-IDF is a numerical statistic that measures how important a word is in a document relative to a collection of documents.
TF-IDF gives high weight to terms that are frequent in one document but rare across the corpus.

- Term Frequency (TF): how often a word appears in a document
- Inverse Document Frequency (IDF): how rare the word is across all documents

Term Frequency:
$\text{tf}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$

Inverse Document Frequency:
$\text{idf}(t, D) = \log\left(\frac{N}{n_t}\right)$

TF-IDF
$\text{tfidf}(t, d, D) = \text{tf}(t, d) \times \text{idf}(t, D)$

Where
- $f_{t,d}$ = count of term $t$ in document $d$
- $N$ = total number of documents
- $n_t$ = number of documents containing term $t$

#### Example

Corpus of 3 documents:

| Doc ID | Text |
|--------|------|
| D1 | “cat sat on mat” |
| D2 | “dog sat on log” |
| D3 | “cat and dog are friends” |

Consider the term **“cat”**:

- Appears in **D1** and **D3** → $n_{\text{cat}} = 2$  
- Total documents $N = 3$
- $\text{idf}(\text{cat}) = \log(3/2) \approx 0.176$

TF values:

- D1: $1/4 = 0.25$
- D2: $0/4 = 0.00$
- D3: $1/5 = 0.20$

TF-IDF table:

| Document | tf("cat") | idf("cat") | tf-idf("cat") |
|---------|-----------|------------|----------------|
| D1 | 0.25 | 0.176 | ≈ 0.044 |
| D2 | 0.00 | 0.176 | 0.000 |
| D3 | 0.20 | 0.176 | ≈ 0.035 |

In [ ]:
# 2. Convert the documents into TF-IDF vectors using scikit-learn.
vectorizer = TfidfVectorizer()
embeddings = vectorizer.fit_transform(
    # ---- YOUR CODE GOES HERE ---
)
embeddings.shape

In [ ]:
# 3. Build a nearest neighbor index to enable similarity search over the vectors.
nn = NearestNeighbors(n_neighbors=3, metric='cosine')
nn.fit(
    # ---- YOUR CODE GOES HERE ---
)

In [ ]:
# 4. Implement a search function to retrieve the most similar documents for a given query.
def search(query, k=3):
    
    # Vectorize the query
    vec = # ---- YOUR CODE GOES HERE ---
    
    # Calculate distances between query and data vectors
    distances, indices = nn.kneighbors(vec, n_neighbors=k)
    
    # Return nearest text chunks and their distance w.r.t. the query
    return pd.DataFrame({
        'result': df.iloc[indices[0]]['text'].values,
        'distance': distances[0]
    })

In [ ]:
# Search the vector database
search("How do I build an API?")

In [ ]:
# 5. Save the vectorizer, index, and data for later use.
joblib.dump(vectorizer, "./assets/vectorizer.pkl")
joblib.dump(nn, "./assets/nn.pkl")
df.to_csv("./assets/texts.csv", index=False)

## **TASK: Deploy Vector Database**

In this task, you will deploy your TF-IDF-based vector database as a web API using FastAPI.

Steps:
1. Load the saved TF-IDF vectorizer, nearest neighbor index, and text data.
2. Set up a FastAPI application to serve search requests.
3. Implement an endpoint that accepts a query and returns the most similar documents from the database.
4. Run the API server with Uvicorn to make your vector search accessible over HTTP.

This workflow demonstrates how to turn a local vector database into a deployable service, enabling integration with other applications for semantic search and retrieval.

In [ ]:
# 1. Load the saved TF-IDF vectorizer, nearest neighbor index, and text data.
vectorizer = joblib.load("./assets/vectorizer.pkl")
nn = joblib.load("./assets/nn.pkl")
df = pd.read_csv("./assets/texts.csv")

In [ ]:
# 2. Set up a FastAPI application to serve search requests.
app = FastAPI()

In [ ]:
# 3. Implement an endpoint that accepts a query and returns the most similar documents from the database.
@app.get("/search")
def search(q: str, k: int = 5):
    # ---- YOUR CODE GOES HERE ---

In [ ]:
# 4. Run the API server with Uvicorn to make your vector search accessible over HTTP.
config = Config(app=app, host="localhost", port=8080, log_level="info")
server = Server(config)

await server.serve()

## **TASK: Deploy a Semantic Search API with FAISS and Sentence Transformers**

In this task, you will deploy a semantic search API using FAISS and Sentence Transformers to enable fast and meaningful retrieval over large text collections.

Steps:
1. Load and chunk a collection of text documents (e.g., essays).
2. Generate dense vector embeddings for each chunk using a pre-trained Sentence Transformer model.
3. Build a FAISS index for efficient similarity search over the embeddings.
4. Implement a retrieval function to find the most relevant text chunks for a given query.
5. Optionally, deploy the retrieval system as a web API for integration with other applications.

This workflow demonstrates how to build and deploy a scalable semantic search service using modern embedding models and vector databases.

In [ ]:
# 1. Load and chunk a collection of text documents (e.g., essays)
files = glob.glob("./data/PaulGrahamEssays/*.txt")
texts = []

for f in files:
    with open(f, "r", encoding="utf-8") as fp:
        texts.append(fp.read())

print(f"Loaded {len(texts)} essays.")

In [ ]:
# 2. Chunk each document into smaller pieces for embedding
def chunk_text(text, chunk_size=400, overlap=80):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = # ---- YOUR CODE GOES HERE ---
        chunks.append(
            # ---- YOUR CODE GOES HERE ---
        )
        start += # ---- YOUR CODE GOES HERE ---
    return chunks

In [ ]:
# 3 Create a DataFrame of all text chunks and their source essay IDs
all_chunks = []
essay_ids = []

for idx, t in enumerate(texts):
    chunks = # ---- YOUR CODE GOES HERE ---
    all_chunks.extend(chunks)
    essay_ids.extend([idx] * len(chunks))

df = pd.DataFrame({
    "essay_id": essay_ids,
    "chunk": all_chunks
})

df

In [ ]:
# 4. Generate dense vector embeddings for each chunk using Sentence Transformers
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df['chunk'].tolist(), show_progress_bar=True)
embeddings.shape

In [ ]:
# 5. Build a FAISS index for efficient similarity search over the embeddings
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)

index.add(np.array(embeddings).astype("float32"))
print("FAISS index contains:", index.ntotal, "vectors")

In [ ]:
# 6. Implement a retrieval function to find the most relevant text chunks for a given query
def retrieve(query, k=5):
    q_emb = # ---- YOUR CODE GOES HERE ---
    distances, indices = # ---- YOUR CODE GOES HERE ---

    results = []
    for score, idx in zip(distances[0], indices[0]):
        results.append({
            "score": float(score),
            "chunk": df.iloc[idx]["chunk"],
            "essay_id": int(df.iloc[idx]["essay_id"])
        })
    return results

In [ ]:
# 7. Test the retrieval function with a sample query
results = retrieve("Why do startups innovate faster?", k=3)
for r in results:
    print("---")
    print("Score:", r["score"])
    print(r["chunk"])